# Episode 13: The Discussion Adapter

Two adapters down. Both were two-party. The `discussion` adapter opens the network to **3, 4, or more agents** — a panel, in a fixed speaking order.

**In this episode you'll build:** a three-agent round-robin debate.

## What `discussion` does

`discussion` is an **N-party round-robin** channel:

- 2 or more participants.
- They speak in a **fixed, cycling order** — the creator first, then the targets in order.
- The adapter enforces "wait your turn" — an out-of-turn send is rejected.
- It **never auto-closes**; you cap it like a `conversation`.

It's the modern replacement for the classic `RoundRobinPattern`. Use it for debates, brainstorming panels, or round-robin reviewers.

## Setup

`ORDERING_ROUND_ROBIN` is the (currently only) ordering. We reuse the WAL-cap helper from Episode 12.

We also quiet one logger: a capable model will occasionally try to speak out of turn, and the adapter correctly rejects it — harmless, but noisy. (See *Additional content*.)

In [ ]:
from dotenv import load_dotenv

load_dotenv()

import asyncio
import logging

from autogen.beta import Agent
from autogen.beta.config import OpenAIConfig
from autogen.beta.knowledge import MemoryKnowledgeStore
from autogen.beta.network import (
    EV_TEXT, ORDERING_ROUND_ROBIN, Hub, HubClient, LocalLink, Passport, Resume,
)

# A capable model occasionally tries to speak out of turn; the adapter rejects
# it harmlessly. Quiet that warning so the demo output stays clean.
logging.getLogger("autogen.beta.network").setLevel(logging.CRITICAL)

config = OpenAIConfig(model="gpt-4.1-mini", temperature=0)


async def wait_for_text_count(hub, channel_id, expected, timeout=200.0):
    deadline = asyncio.get_event_loop().time() + timeout
    while asyncio.get_event_loop().time() < deadline:
        wal = await hub.read_wal(channel_id)
        if sum(1 for e in wal if e.event_type == EV_TEXT) >= expected:
            return
        await asyncio.sleep(0.1)
    raise asyncio.TimeoutError("count not reached")


## A three-way debate

Three agents — an optimist, a realist, a skeptic — debate one topic. The speaking order is fixed: `alice → bob → carol → alice → ...`. We cap it at 6 messages (two full cycles).

In [1]:
hub = await Hub.open(MemoryKnowledgeStore(), ttl_sweep_interval=0)
link = LocalLink(hub)
a_hc, b_hc, c_hc = (HubClient(link, hub=hub) for _ in range(3))

alice = await a_hc.register(
    Agent("alice", prompt="You are an optimist about technology. "
          "Add one short sentence to the debate. No name prefix.", config=config),
    Passport(name="alice"), Resume(), attach_plugin=False,
)
bob = await b_hc.register(
    Agent("bob", prompt="You are a pragmatic realist. "
          "Add one short sentence to the debate. No name prefix.", config=config),
    Passport(name="bob"), Resume(), attach_plugin=False,
)
carol = await c_hc.register(
    Agent("carol", prompt="You are a thoughtful skeptic. "
          "Add one short sentence to the debate. No name prefix.", config=config),
    Passport(name="carol"), Resume(), attach_plugin=False,
)

# discussion = N-party round-robin. Order = creator, then targets in order.
channel = await alice.open(
    type="discussion",
    target=[bob.agent_id, carol.agent_id],
    knobs={"ordering": ORDERING_ROUND_ROBIN},
)
await channel.send("Topic: should every developer learn Rust?")

# Stop after 6 messages = two full alice/bob/carol cycles.
await wait_for_text_count(hub, channel.channel_id, expected=6)
await channel.close(reason="cap_reached")

names = {alice.agent_id: "alice", bob.agent_id: "bob", carol.agent_id: "carol"}
for env in await hub.read_wal(channel.channel_id):
    if env.event_type == EV_TEXT:
        print(f"{names[env.sender_id]}: {env.event_data['text']}")

for hc in (a_hc, b_hc, c_hc):
    await hc.close()
await hub.close()


alice: Topic: should every developer learn Rust?
bob: While Rust offers many benefits, it's more practical for developers to choose languages based on their specific project needs and goals.
carol: Not every developer may need Rust, but understanding its principles can improve overall programming skills.
alice: Rust's focus on safety and performance inspires better coding practices across many programming languages.
bob: While Rust offers valuable lessons, practical needs and project requirements should guide a developer's learning priorities.
carol: Isn't it possible that emphasizing Rust might overshadow other important languages and skills developers need?


## How turn-skipping works

When `alice` speaks, the message fans out to `bob` and `carol` at the same time. Both handlers wake up — but each first asks the Hub "is it my turn?". Only `bob` (the next speaker) gets a yes and runs his LLM; `carol` skips without an LLM call. No wasted work, no race.

The order in `target=[bob.agent_id, carol.agent_id]` is significant — it *is* the rotation after the creator.

## Additional content

**The out-of-turn warning.** A capable model sometimes tries to post an extra message in its turn. The adapter rejects the out-of-turn envelope and the round-robin continues correctly — it's harmless, which is why we quieted the log. The docs describe a custom handler that prevents it entirely.

**Discussion never auto-closes from a slow speaker.** A stalled participant gets hidden for a cycle but the panel keeps going — one slow agent shouldn't kill the discussion.

**Need conditions?** If turn-taking should depend on *what was said* ("if bob mentions security, jump to the security expert"), `discussion` is the wrong tool. That's `workflow` — next episode.

## What's next

`discussion` gives a *fixed* order. Episode 14's `workflow` adapter lets a declarative graph decide who speaks next — conditionally.